# Week 2 Day 06 — LLM API Experiments

## Objectives

- Call an LLM API from Jupyter
- Load the API key using an environment variable
- Sweep temperature from 0 to 1
- Count tokens for several inputs
- Relate token usage to cost
- Trigger and document a hallucination
- Improve the prompt to reduce hallucination

In [ ]:
%pip install openai python-dotenv tiktoken

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY not found in .env")

print("API key loaded successfully.")

## 1. Load the API key securely

The real API key is stored in `.env`.

Never hard-code the API key inside the notebook.

## 2. Create the LLM client

Create the OpenAI client using the API key loaded from the environment.

In [ ]:
import openai

client = openai.OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=api_key
)

print("Client created successfully.")

## 3. First real LLM API call

Send a simple prompt to the model and display its response.

In [ ]:
prompt = "Explain what an API is to a beginner in three sentences."
Model = "openai/gpt-oss-20b"
response = client.responses.create(
    model=Model,
    input=prompt
)

print(response.output_text)

## 2. Temperature Sweep

Temperature controls the randomness/variation of the model's output.

We will use the same prompt with temperatures from 0 to 1 and compare the responses.

In [ ]:
prompt = "Write a creative description of a rainy evening in Chennai in 50 words."

temperatures = [0, 0.2, 0.4, 0.6, 0.8, 1.0]

for temperature in temperatures:
    response = client.responses.create(
        model=Model,
        input=prompt,
        temperature=temperature
    )

    print("=" * 60)
    print("Temperature:", temperature)
    print(response.output_text)

## 3. Token Counting and Cost

LLMs process text as tokens rather than ordinary words.

More tokens generally means more usage and, for paid APIs, potentially higher cost.

In [ ]:
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")

inputs = [
    "Hello world!",
    "Explain Python functions to a beginner.",
    "Large language models generate text one token at a time.",
    "PostgreSQL, Redis, FastAPI, Docker, and Python are backend technologies."
]

for text in inputs:
    token_count = len(encoding.encode(text))

    print("-" * 60)
    print("Text:", text)
    print("Token count:", token_count)

### Cost relationship

For a paid API:

**Input cost = input tokens × input token price**

**Output cost = output tokens × output token price**

The exact price depends on the model.

## 4. Trigger a Hallucination

We will give the model a false premise and observe whether it invents information.

In [ ]:
prompt = """
Who was Dr. Arvind Ramanathan, the famous Indian computer scientist
who invented the Ramanathan Algorithm in 1987?

Give his biography, university, awards, and major publications.
"""

response = client.responses.create(
    model=Model,
    input=prompt,
    reasoning={
        "effort": "medium"
    }
)

print(response.output_text)



## Hallucination Observation

Review the response above.

Look for:

- Invented biography
- Invented university
- Invented awards
- Invented publications
- Unsupported dates or achievements

A fluent and confident answer does not necessarily mean the information is true.

## 5. Fix the Hallucination With Better Prompting

Now explicitly tell the model not to assume the claim is true or invent facts.

In [ ]:
prompt = """
Analyze this claim carefully:

Dr. Arvind Ramanathan was a famous Indian computer scientist
who invented the Ramanathan Algorithm in 1987.

Do not assume this claim is true.
Do not invent facts.

If you cannot verify that this person or algorithm exists,
say so clearly.

Separate known information from assumptions in the question.
"""

response = client.responses.create(
    model=Model,
    input=prompt,
    temperature=0
)

print(response.output_text)

## Hallucination Experiment — Final Observation

The improved prompt produced a safer response.

The model did not accept the fictional premise as fact. Instead, it:

- Identified the claim as unverified.
- Separated known information from assumptions.
- Avoided inventing a biography, university, awards, or publications.
- Clearly stated that the claim could not be confirmed.
- Recommended checking primary sources for verification.

### Conclusion

Better prompting can reduce hallucination by explicitly instructing the model
to question assumptions, avoid inventing facts, and communicate uncertainty.

However, this does not guarantee factual accuracy. Important information
should still be verified using reliable external sources.